# Lab 06 – Text Representation Techniques
## One-Hot Encoding, Bag of Words (BOW) and TF-IDF

**Requirements:** Python 3.x, `numpy`, `scikit-learn`

In [1]:
# !pip install numpy scikit-learn
import math
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
np.set_printoptions(precision=4, suppress=True)

---
# Experiment 1: One-Hot Encoding
**Aim:** Convert each unique word into a binary vector.

In [2]:
documents = ["I like NLP", "I like AI", "NLP is interesting"]

# Build vocabulary (order of first appearance)
vocabulary = []
for d in documents:
    for w in d.split():
        if w not in vocabulary:
            vocabulary.append(w)
print("Vocabulary:", vocabulary)

# One-hot vector for every vocabulary word
for w in vocabulary:
    vector = [1 if v == w else 0 for v in vocabulary]
    print(f"{w:12} {vector}")

# Each document as a sequence of one-hot vectors
print("\nDocument representation:")
for d in documents:
    print(f"\n'{d}'")
    for w in d.split():
        vector = [1 if v == w else 0 for v in vocabulary]
        print(f"  {w:12} {vector}")

Vocabulary: ['I', 'like', 'NLP', 'AI', 'is', 'interesting']
I            [1, 0, 0, 0, 0, 0]
like         [0, 1, 0, 0, 0, 0]
NLP          [0, 0, 1, 0, 0, 0]
AI           [0, 0, 0, 1, 0, 0]
is           [0, 0, 0, 0, 1, 0]
interesting  [0, 0, 0, 0, 0, 1]

Document representation:

'I like NLP'
  I            [1, 0, 0, 0, 0, 0]
  like         [0, 1, 0, 0, 0, 0]
  NLP          [0, 0, 1, 0, 0, 0]

'I like AI'
  I            [1, 0, 0, 0, 0, 0]
  like         [0, 1, 0, 0, 0, 0]
  AI           [0, 0, 0, 1, 0, 0]

'NLP is interesting'
  NLP          [0, 0, 1, 0, 0, 0]
  is           [0, 0, 0, 0, 1, 0]
  interesting  [0, 0, 0, 0, 0, 1]


**Observation**
- Each vector has exactly one `1`; all other values are `0`.
- Vector length = vocabulary size (6).
- Frequency, word order and semantic similarity are not represented.

---
# Experiment 2: Bag of Words (BOW)
**Aim:** Represent documents using word counts.

In [3]:
documents = ["I like NLP", "I like AI", "NLP is interesting"]

vocabulary = []
for d in documents:
    for w in d.lower().split():
        if w not in vocabulary:
            vocabulary.append(w)
print("Vocabulary:", vocabulary)

for i, d in enumerate(documents, 1):
    words = d.lower().split()
    row = [words.count(w) for w in vocabulary]
    print(f"D{i}:", row)

# Repetition example
text = "AI is good. AI is powerful. AI is useful."
words = [w.strip(".").lower() for w in text.split()]
vocab = []
for w in words:
    if w not in vocab:
        vocab.append(w)
print("\nRepetition example:")
print("Vocabulary:", vocab)
print("BOW:       ", [words.count(w) for w in vocab])

Vocabulary: ['i', 'like', 'nlp', 'ai', 'is', 'interesting']
D1: [1, 1, 1, 0, 0, 0]
D2: [1, 1, 0, 1, 0, 0]
D3: [0, 0, 1, 0, 1, 1]

Repetition example:
Vocabulary: ['ai', 'is', 'good', 'powerful', 'useful']
BOW:        [3, 3, 1, 1, 1]


**Observation**
- BOW records word frequency.
- Word order is ignored.
- Repeated words produce larger counts ("ai" and "is" → 3).

---
# Experiment 3: Manual TF, DF, IDF and TF-IDF
**Aim:** Calculate TF, DF, IDF and TF-IDF from a small corpus.

- TF(t, d) = count of t in d / total terms in d
- DF(t) = number of documents containing t
- IDF(t) = ln(N / DF)
- TF-IDF = TF × IDF

In [4]:
documents = ["cat dog", "cat mouse", "dog mouse"]
docs = [d.split() for d in documents]
vocabulary = sorted(set(w for d in docs for w in d))
N = len(docs)
print("Vocabulary:", vocabulary)

for term in vocabulary:
    df = sum(1 for d in docs if term in d)
    idf = math.log(N / df)
    print(f"{term}: DF={df}, IDF={idf:.3f}")

print("\nTF matrix:")
for i, d in enumerate(docs, 1):
    print(f"D{i}:", [round(d.count(t) / len(d), 4) for t in vocabulary])

print("\nTF-IDF matrix:")
for i, d in enumerate(docs, 1):
    row = []
    for term in vocabulary:
        tf = d.count(term) / len(d)
        df = sum(1 for x in docs if term in x)
        idf = math.log(N / df)
        row.append(tf * idf)
    print(f"D{i}:", [round(x, 4) for x in row])

Vocabulary: ['cat', 'dog', 'mouse']
cat: DF=2, IDF=0.405
dog: DF=2, IDF=0.405
mouse: DF=2, IDF=0.405

TF matrix:
D1: [0.5, 0.5, 0.0]
D2: [0.5, 0.0, 0.5]
D3: [0.0, 0.5, 0.5]

TF-IDF matrix:
D1: [0.2027, 0.2027, 0.0]
D2: [0.2027, 0.0, 0.2027]
D3: [0.0, 0.2027, 0.2027]


**Observation**
- Every term appears in 2 of 3 documents, so IDF = ln(3/2) ≈ 0.405 for all terms.
- A present term has TF = 1/2, so TF-IDF = 0.5 × 0.405 ≈ 0.2027; absent terms get 0.

---
# Experiment 4: TF-IDF Using Scikit-Learn
**Aim:** Generate a TF-IDF document-term matrix automatically.

In [5]:
documents = ["I love machine learning", "I love deep learning", "I study machine learning"]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

print("Features:", vectorizer.get_feature_names_out())
print("TF-IDF matrix:")
print(X.toarray())
print("Shape:", X.shape)

Features: ['deep' 'learning' 'love' 'machine' 'study']
TF-IDF matrix:
[[0.     0.4813 0.6198 0.6198 0.    ]
 [0.7203 0.4254 0.5478 0.     0.    ]
 [0.     0.4254 0.     0.5478 0.7203]]
Shape: (3, 5)


**Why sklearn values differ from the hand formula** – by default `TfidfVectorizer`:
1. uses raw counts as TF (not count / document length),
2. uses smoothed IDF: `idf = ln((1 + N) / (1 + DF)) + 1`,
3. L2-normalises each row,
4. drops single-character tokens such as "I" (default `token_pattern`).

The cell below reproduces sklearn's output manually to prove this.

In [6]:
features = list(vectorizer.get_feature_names_out())
tokenized = [[w for w in d.lower().split() if len(w) > 1] for d in documents]
N = len(tokenized)

manual = []
for d in tokenized:
    row = []
    for t in features:
        tf = d.count(t)
        df = sum(1 for x in tokenized if t in x)
        idf = math.log((1 + N) / (1 + df)) + 1
        row.append(tf * idf)
    row = np.array(row)
    manual.append(row / np.linalg.norm(row))   # L2 normalisation
manual = np.array(manual)

print("Manual (sklearn convention):\n", manual)
print("Matches sklearn:", np.allclose(manual, X.toarray()))

Manual (sklearn convention):
 [[0.     0.4813 0.6198 0.6198 0.    ]
 [0.7203 0.4254 0.5478 0.     0.    ]
 [0.     0.4254 0.     0.5478 0.7203]]
Matches sklearn: True


**Observation**
- The result is a weighted document-term matrix of shape (3 documents × 5 features).
- "learning" appears in every document, so it receives the lowest weight in each row.
- Scikit-Learn uses smoothing and normalisation, so its numbers differ from the simple hand formula.

---
# Experiment 5: Integrated Experiment – One-Hot vs BOW vs TF-IDF
**Aim:** Apply all three techniques to the same corpus and compare outputs.

In [7]:
documents = [
    "natural language processing",
    "machine learning for language",
    "deep learning for natural language",
    "machine learning models",
]

# One-Hot vocabulary representation
vocab = sorted(set(" ".join(documents).lower().split()))
print("Vocabulary:", vocab)
for w in vocab:
    print(f"{w:12}", [1 if v == w else 0 for v in vocab])

# BOW
bow = CountVectorizer()
X_bow = bow.fit_transform(documents)
print("\nBOW features:", bow.get_feature_names_out())
print(X_bow.toarray())

# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(documents)
print("\nTF-IDF features:", tfidf.get_feature_names_out())
print(X_tfidf.toarray())

print("\nDimensions -> One-Hot:", (len(vocab), len(vocab)),
      "| BOW:", X_bow.shape, "| TF-IDF:", X_tfidf.shape)

Vocabulary: ['deep', 'for', 'language', 'learning', 'machine', 'models', 'natural', 'processing']
deep         [1, 0, 0, 0, 0, 0, 0, 0]
for          [0, 1, 0, 0, 0, 0, 0, 0]
language     [0, 0, 1, 0, 0, 0, 0, 0]
learning     [0, 0, 0, 1, 0, 0, 0, 0]
machine      [0, 0, 0, 0, 1, 0, 0, 0]
models       [0, 0, 0, 0, 0, 1, 0, 0]
natural      [0, 0, 0, 0, 0, 0, 1, 0]
processing   [0, 0, 0, 0, 0, 0, 0, 1]

BOW features: ['deep' 'for' 'language' 'learning' 'machine' 'models' 'natural'
 'processing']
[[0 0 1 0 0 0 1 1]
 [0 1 1 1 1 0 0 0]
 [1 1 1 1 0 0 1 0]
 [0 0 0 1 1 1 0 0]]

TF-IDF features: ['deep' 'for' 'language' 'learning' 'machine' 'models' 'natural'
 'processing']
[[0.     0.     0.4481 0.     0.     0.     0.5535 0.702 ]
 [0.     0.5496 0.4449 0.4449 0.5496 0.     0.     0.    ]
 [0.5718 0.4509 0.365  0.365  0.     0.     0.4509 0.    ]
 [0.     0.     0.     0.4481 0.5535 0.702  0.     0.    ]]

Dimensions -> One-Hot: (8, 8) | BOW: (4, 8) | TF-IDF: (4, 8)


**Observation**
- One-Hot produces one binary vector per vocabulary word (8 × 8), with no document-level information.
- BOW shows how many times each word occurs in each document (4 × 8).
- TF-IDF gives the same shape as BOW, but rarer words such as "processing", "deep" and "models" get higher weights than common words such as "language" and "learning".

---
# Student Exercises

## Exercise 1: One-Hot vectors for a corpus with at least 8 unique words

In [8]:
corpus = [
    "students read books in the library",
    "teachers write notes for students",
    "the library opens early",
]

vocab = []
for d in corpus:
    for w in d.lower().split():
        if w not in vocab:
            vocab.append(w)

print("Number of unique words:", len(vocab))
assert len(vocab) >= 8

print(f"\n{'word':10}", "one-hot vector")
for w in vocab:
    print(f"{w:10}", [1 if v == w else 0 for v in vocab])

# Same result as an identity matrix
one_hot = np.eye(len(vocab), dtype=int)
print("\nOne-hot matrix shape:", one_hot.shape)

Number of unique words: 12

word       one-hot vector
students   [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
read       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
books      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
in         [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
the        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
library    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
teachers   [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
write      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
notes      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
for        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
opens      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
early      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

One-hot matrix shape: (12, 12)


## Exercise 2: BOW matrix for three documents + manual verification

In [9]:
docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
]

# Manual BOW
vocab = sorted(set(w for d in docs for w in d.split()))
manual_bow = [[d.split().count(w) for w in vocab] for d in docs]

print("Vocabulary:", vocab)
print(f"{'':4}" + "".join(f"{w:>8}" for w in vocab))
for i, row in enumerate(manual_bow, 1):
    print(f"D{i}: " + "".join(f"{c:>8}" for c in row))

# Verify against sklearn's CountVectorizer
cv = CountVectorizer()
X = cv.fit_transform(docs).toarray()
print("\nsklearn vocabulary:", list(cv.get_feature_names_out()))
print("Manual BOW == CountVectorizer:", np.array_equal(np.array(manual_bow), X))

Vocabulary: ['cat', 'chased', 'dog', 'log', 'mat', 'on', 'sat', 'the']
         cat  chased     dog     log     mat      on     sat     the
D1:        1       0       0       0       1       1       1       2
D2:        0       0       1       1       0       1       1       2
D3:        1       1       1       0       0       0       0       2

sklearn vocabulary: ['cat', 'chased', 'dog', 'log', 'mat', 'on', 'sat', 'the']
Manual BOW == CountVectorizer: True


**Manual check:** In D1 "the cat sat on the mat" → the = 2, cat = 1, sat = 1, on = 1, mat = 1, every other word = 0. This matches the first row. Each row also sums to the number of words in that document (6, 6, 5).

## Exercise 3: TF, DF, IDF and TF-IDF for four documents
Using the lab formulae: TF = count / total terms, IDF = ln(N / DF).

In [10]:
documents = [
    "the sun is hot",
    "the sun is bright",
    "the moon is bright",
    "stars shine at night",
]
docs = [d.split() for d in documents]
vocab = sorted(set(w for d in docs for w in d))
N = len(docs)

# DF and IDF
df = {t: sum(1 for d in docs if t in d) for t in vocab}
idf = {t: math.log(N / df[t]) for t in vocab}

print(f"{'term':8}{'DF':>4}{'IDF':>8}")
for t in vocab:
    print(f"{t:8}{df[t]:>4}{idf[t]:>8.4f}")

# TF matrix
tf = [[d.count(t) / len(d) for t in vocab] for d in docs]
# TF-IDF matrix
tfidf_m = [[tf[i][j] * idf[t] for j, t in enumerate(vocab)] for i in range(N)]

def show(title, M):
    print(f"\n{title}")
    print(f"{'':4}" + "".join(f"{t:>8}" for t in vocab))
    for i, row in enumerate(M, 1):
        print(f"D{i}: " + "".join(f"{v:>8.4f}" for v in row))

show("TF matrix", tf)
show("TF-IDF matrix", tfidf_m)

term      DF     IDF
at         1  1.3863
bright     2  0.6931
hot        1  1.3863
is         3  0.2877
moon       1  1.3863
night      1  1.3863
shine      1  1.3863
stars      1  1.3863
sun        2  0.6931
the        3  0.2877

TF matrix
          at  bright     hot      is    moon   night   shine   stars     sun     the
D1:   0.0000  0.0000  0.2500  0.2500  0.0000  0.0000  0.0000  0.0000  0.2500  0.2500
D2:   0.0000  0.2500  0.0000  0.2500  0.0000  0.0000  0.0000  0.0000  0.2500  0.2500
D3:   0.0000  0.2500  0.0000  0.2500  0.2500  0.0000  0.0000  0.0000  0.0000  0.2500
D4:   0.2500  0.0000  0.0000  0.0000  0.0000  0.2500  0.2500  0.2500  0.0000  0.0000

TF-IDF matrix
          at  bright     hot      is    moon   night   shine   stars     sun     the
D1:   0.0000  0.0000  0.3466  0.0719  0.0000  0.0000  0.0000  0.0000  0.1733  0.0719
D2:   0.0000  0.1733  0.0000  0.0719  0.0000  0.0000  0.0000  0.0000  0.1733  0.0719
D3:   0.0000  0.1733  0.0000  0.0719  0.3466  0.0000  0.0000  0

**Worked example (term "hot" in D1):**
TF = 1/4 = 0.25, DF = 1, IDF = ln(4/1) = 1.3863, TF-IDF = 0.25 × 1.3863 = 0.3466.

**Term "is" in D1:** TF = 0.25, DF = 3, IDF = ln(4/3) = 0.2877, TF-IDF = 0.0719.

"hot" is unique to D1, so it gets a much higher weight than "is", which appears in three documents.

## Exercise 4: Same corpus through BOW and TF-IDF – compare the matrices

In [11]:
corpus = [
    "the movie was good and the acting was good",
    "the movie was bad and the story was bad",
    "the acting was excellent",
]

cv = CountVectorizer()
tv = TfidfVectorizer()
X_bow = cv.fit_transform(corpus).toarray()
X_tfidf = tv.fit_transform(corpus).toarray()
features = cv.get_feature_names_out()

print(f"{'term':10}" + "".join(f"{'BOW D'+str(i+1):>9}" for i in range(3))
      + "".join(f"{'TFIDF D'+str(i+1):>10}" for i in range(3)))
for j, t in enumerate(features):
    print(f"{t:10}" + "".join(f"{X_bow[i, j]:>9}" for i in range(3))
          + "".join(f"{X_tfidf[i, j]:>10.3f}" for i in range(3)))

# Highest-weighted word per document under each scheme
print()
for i in range(len(corpus)):
    bow_top = [str(f) for f in features[X_bow[i] == X_bow[i].max()]]
    print(f"D{i+1}: top BOW word(s) = {bow_top!s:28} "
          f"top TF-IDF word = {features[X_tfidf[i].argmax()]!r}")

term         BOW D1   BOW D2   BOW D3  TFIDF D1  TFIDF D2  TFIDF D3
acting            1        0        1     0.260     0.000     0.504
and               1        1        0     0.260     0.254     0.000
bad               0        2        0     0.000     0.669     0.000
excellent         0        0        1     0.000     0.000     0.663
good              2        0        0     0.685     0.000     0.000
movie             1        1        0     0.260     0.254     0.000
story             0        1        0     0.000     0.334     0.000
the               2        2        1     0.405     0.395     0.391
was               2        2        1     0.405     0.395     0.391

D1: top BOW word(s) = ['good', 'the', 'was']       top TF-IDF word = 'good'
D2: top BOW word(s) = ['bad', 'the', 'was']        top TF-IDF word = 'bad'
D3: top BOW word(s) = ['acting', 'excellent', 'the', 'was'] top TF-IDF word = 'excellent'


**Comparison**
- Both matrices have the same shape and the same zero pattern (a word absent from a document is 0 in both).
- BOW: "the" and "was" tie with "good"/"bad" for the highest count in D1 and D2, even though they carry no meaning. In D3 every word has count 1, so BOW cannot say which word matters.
- TF-IDF: words that appear in every document ("the", "was") are pushed down, while "good", "bad" and "excellent" become the top terms, which are the words that actually tell the reviews apart.
- TF-IDF rows are L2-normalised (each row has length 1), so long and short documents are comparable; BOW values grow with document length.

## Exercise 5: Why TF-IDF can be more informative than raw BOW

1. **Down-weights common words.** In BOW, frequent words like "the", "is", "was" dominate the vector. IDF gives words that occur in many documents a low weight (zero in the basic formula if a word is in every document), so they stop dominating.
2. **Up-weights discriminative words.** Words that appear in only a few documents get a high IDF, so the features that separate one document from another stand out (see "good"/"bad"/"excellent" in Exercise 4).
3. **Reduces document-length bias.** TF is relative to document length (and sklearn also L2-normalises), so a long document does not look more important just because it has more words.
4. **Better similarity and retrieval.** Because shared stop-words contribute little, cosine similarity between TF-IDF vectors reflects shared *content* words, which improves search ranking and classification.

The demo below shows this with a query.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "the cat is on the mat and the cat is happy",
    "the dog is in the house and the dog is happy",
    "quantum physics explains the behaviour of particles",
]
query = ["the cat"]

for name, vec in [("BOW", CountVectorizer()), ("TF-IDF", TfidfVectorizer())]:
    M = vec.fit_transform(docs)
    q = vec.transform(query)
    sims = cosine_similarity(q, M)[0]
    print(f"{name:7} similarity of 'the cat' to D1, D2, D3:", np.round(sims, 3))

BOW     similarity of 'the cat' to D1, D2, D3: [0.772 0.463 0.267]
TF-IDF  similarity of 'the cat' to D1, D2, D3: [0.739 0.254 0.119]


With BOW the query "the cat" looks fairly similar to the dog document as well, because "the" is shared and frequent. With TF-IDF, "the" (present in every document) gets little weight, so the cat document stands out much more clearly against the others.

## Exercise 6: When One-Hot is sufficient vs when TF-IDF is preferable

**One-Hot Encoding is sufficient:** representing a small set of categorical labels or tokens where frequency does not matter. Examples: encoding intent/class labels for a chatbot classifier (e.g. `greeting`, `booking`, `cancel`), encoding a small fixed command vocabulary for a voice assistant ("start", "stop", "pause"), or as the input index for an embedding layer in a neural network, where each word ID is one-hot before being mapped to a dense vector.

**TF-IDF is preferable:** tasks where whole documents must be compared or ranked by content. Examples: search engines / information retrieval (rank documents for a query), spam detection, news topic classification and keyword extraction. Here, distinguishing rare, informative words from common ones is essential.

In [13]:
# One-Hot: categorical intent labels
intents = ["greeting", "booking", "cancel"]
for intent in intents:
    print(f"{intent:9}", [1 if i == intent else 0 for i in intents])

# TF-IDF: tiny search engine
articles = [
    "football match ends in a draw after extra time",
    "new smartphone launched with a faster processor",
    "stock market rises as tech shares gain",
    "team wins the football championship final",
]
tv = TfidfVectorizer(stop_words="english")
A = tv.fit_transform(articles)
q = tv.transform(["football final"])
scores = cosine_similarity(q, A)[0]
print("\nSearch results for 'football final':")
for idx in scores.argsort()[::-1]:
    print(f"  {scores[idx]:.3f}  {articles[idx]}")

greeting  [1, 0, 0]
booking   [0, 1, 0]
cancel    [0, 0, 1]

Search results for 'football final':
  0.592  team wins the football championship final
  0.206  football match ends in a draw after extra time
  0.000  stock market rises as tech shares gain
  0.000  new smartphone launched with a faster processor


---
# Conclusion
One-Hot Encoding is a simple binary representation that identifies words but ignores frequency and order. BOW captures word frequency but still ignores order and treats every word as equally important. TF-IDF weights words by how frequent they are in a document and how rare they are across the corpus, so it highlights the terms that distinguish documents. The right technique depends on the task: One-Hot for categorical inputs, BOW for simple baselines, and TF-IDF for retrieval and classical text classification.